# HOAPS Compression-Ratio Contributor Analysis

This notebook quantifies which design choices contribute most to compression ratio (CR) for the HOAPS water-vapor field codec.

We analyze:

1. Dataset size and missing-value structure.
2. Missing-mask representations.
3. Candidate reconstructed-neighbor predictors and residual statistics.
4. Entropy modes used by the project (`RAW`, `RANGE`, and `CTX`).
5. End-to-end compression/decompression, bounded error, timing, and achieved CR.

The data is loaded from the challenge S3 object store with `open_remote_dataset`.

In [ ]:
from pathlib import Path
import sys
import time
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)
try:
    import seaborn as sns
except ImportError:  # Optional visualization dependency.
    sns = None
import zstandard as zstd
import xarray as xr
from upath import UPath
from utils import open_remote_dataset

# Make the checkout importable when this notebook is run from the repository.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = next(parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "src").exists())
sys.path.insert(0, str(ROOT / "src"))

from hoaps_compressor import HoapsWvpaCodec
from hoaps_compressor.container import read_container
from hoaps_compressor.mask import pack_mask
from hoaps_compressor.model import entropy

BASE_URL = "https://object-store.os-api.cci1.ecmwf.int/esiwacebucket"
remote_data = UPath(BASE_URL)
dataset = open_remote_dataset(
    remote_data / "HOAPS" / "HOAPS_2020-08_6-hourly.nc",
    engine="h5netcdf",
    decode_timedelta=True,
)
da = dataset["wvpa"].sel(time=slice("2020-08-01", "2020-08-07"))
field = np.ascontiguousarray(da.values.astype(np.float32, copy=False))
if field.ndim == 2:
    field = field[None, ...]
mask = ~np.isfinite(field)
valid = ~mask

print(f"data: {remote_data / 'HOAPS' / 'HOAPS_2020-08_6-hourly.nc'}")
print(f"selection: {da.time.values[0]} to {da.time.values[-1]}")
print(f"shape={field.shape}, dtype={field.dtype}, cells={field.size:,}")

## 1. Load and inspect the dataset

The input is a 3-D `float32` field with dimensions `(time, latitude, longitude)`. Missing values are represented by non-finite values and must be encoded losslessly rather than predicted as ordinary numbers.

In [ ]:
summary = pd.DataFrame({
    "metric": ["shape", "dtype", "cells", "valid", "missing", "missing_%", "original_bytes", "min", "max", "mean", "std"],
    "value": [
        str(field.shape), str(field.dtype), field.size, int(valid.sum()), int(mask.sum()),
        100 * mask.mean(), field.nbytes, float(np.nanmin(field)), float(np.nanmax(field)),
        float(np.nanmean(field)), float(np.nanstd(field)),
    ],
})
display(summary)

# Per-time-slice missingness reveals whether temporal mask prediction is plausible.
missing_by_time = pd.Series(mask.mean(axis=(1, 2)), name="missing_fraction")
display(missing_by_time.describe().to_frame())
missing_by_time.plot(title="Missing fraction by time slice", figsize=(8, 3), marker=".")
plt.ylabel("missing fraction")
plt.show()

## 2. Compression-ratio metrics

For original size $S_o$ and compressed size $S_c$:

$$CR = \frac{S_o}{S_c}, \qquad reduction = 100\left(1 - \frac{S_c}{S_o}\right).$$

For valid values, we also measure MAE, RMSE, maximum absolute error, residual range, and residual entropy. Entropy is estimated from the empirical symbol distribution in bits per symbol; the final encoded size also includes headers and coder tables.

In [ ]:
def cr_metrics(original_bytes, compressed_bytes):
    cr = original_bytes / compressed_bytes if compressed_bytes else np.inf
    return {"original_bytes": original_bytes, "compressed_bytes": compressed_bytes,
            "cr": cr, "reduction_pct": 100 * (1 - compressed_bytes / original_bytes)}

def residual_metrics(residuals, elapsed_s=None):
    r = np.asarray(residuals, dtype=np.float64)
    counts = np.unique(r, return_counts=True)[1]
    probabilities = counts / counts.sum()
    entropy_bits = float(-(probabilities * np.log2(probabilities)).sum())
    out = {
        "mae": float(np.mean(np.abs(r))),
        "rmse": float(np.sqrt(np.mean(r ** 2))),
        "max_abs": float(np.max(np.abs(r))),
        "residual_std": float(np.std(r)),
        "residual_range": float(np.max(r) - np.min(r)),
        "entropy_bits_per_symbol": entropy_bits,
    }
    if elapsed_s is not None:
        out["seconds"] = elapsed_s
    return out

def display_metric_table(rows, sort_by=None):
    result = pd.DataFrame(rows)
    if sort_by and sort_by in result:
        result = result.sort_values(sort_by).reset_index(drop=True)
    display(result)
    return result

## 3. Prepare data and handle missing values

Dropping missing records changes the grid and breaks positional reconstruction. Replacing missing values with a constant, mean, median, forward fill, backward fill, or interpolation can improve an ordinary predictor, but it hides the original mask and may introduce false valid data. The codec therefore uses an explicit lossless mask and excludes missing cells from prediction.

In [ ]:
def rle_size(bits):
    bits = np.asarray(bits, dtype=bool).ravel()
    if bits.size == 0:
        return 0
    changes = np.flatnonzero(bits[1:] != bits[:-1]) + 1
    boundaries = np.concatenate(([0], changes, [bits.size]))
    lengths = np.diff(boundaries)
    # HMR1 magic + initial value + LEB128 run lengths.
    varint_bytes = sum(max(1, (int(length).bit_length() + 6) // 7) for length in lengths)
    return 5 + varint_bytes

def mask_candidates(mask):
    candidates = {}
    flat = mask.ravel()
    bitpack = np.packbits(flat, bitorder="little").tobytes()
    rle = pack_mask(mask)
    z = zstd.ZstdCompressor(level=3)
    candidates["bitpack"] = bitpack
    candidates["RLE-or-bitpack"] = rle
    candidates["bitpack+Zstandard"] = z.compress(bitpack)
    candidates["RLE-or-bitpack+Zstandard"] = z.compress(rle)
    # Alternate layout: latitude-major, longitude-major, time-minor.
    lat_lon_time = np.transpose(mask, (1, 2, 0))
    candidates["lat-lon-time bitpack+Zstandard"] = z.compress(
        np.packbits(lat_lon_time.ravel(), bitorder="little").tobytes()
    )
    return pd.DataFrame({"strategy": list(candidates), "bytes": [len(v) for v in candidates.values()]})

mask_table = mask_candidates(mask)
mask_table["share_of_original_%"] = 100 * mask_table["bytes"] / field.nbytes
display(mask_table.sort_values("bytes"))

# Demonstrate the explicit mask invariant.
assert np.array_equal(mask, ~np.isfinite(np.where(mask, np.nan, field)))
print(f"missing cells: {mask.sum():,} ({mask.mean():.1%})")

## 4. Implement candidate predictors

The following predictors are deliberately transparent baselines. They operate on the valid values in scan order so their residual statistics are comparable:

- previous-value and delta prediction;
- moving average;
- linear extrapolation from the two previous values;
- local quadratic extrapolation;
- the project’s reconstructed-neighbor predictor.

For production selection, predictors must be decoder-safe: the decoder can use only metadata, the mask, and values already reconstructed.

In [ ]:
valid_values = field[valid].astype(np.float64)
# Keep experiments bounded and fast while preserving the real distribution.
analysis_values = valid_values[: min(valid_values.size, 300_000)]


def candidate_residuals(values, name):
    values = np.asarray(values, dtype=np.float64)
    residuals = np.empty_like(values)
    residuals[:1] = values[:1] - 32.0
    if name == "previous-value":
        residuals[1:] = values[1:] - values[:-1]
    elif name == "delta":
        residuals[1] = values[1] - values[0]
        residuals[2:] = values[2:] - 2 * values[1:-1] + values[:-2]
    elif name == "moving-average-4":
        for i in range(1, len(values)):
            residuals[i] = values[i] - values[max(0, i - 4):i].mean()
    elif name == "linear-2":
        residuals[1] = values[1] - values[0]
        residuals[2:] = values[2:] - (2 * values[1:-1] - values[:-2])
    elif name == "quadratic-3":
        residuals[1] = values[1] - values[0]
        residuals[2] = values[2] - (2 * values[1] - values[0])
        for i in range(3, len(values)):
            residuals[i] = values[i] - (3 * values[i - 1] - 3 * values[i - 2] + values[i - 3])
    else:
        raise ValueError(name)
    return residuals

predictor_names = ["previous-value", "delta", "moving-average-4", "linear-2", "quadratic-3"]
rows = []
for name in predictor_names:
    started = time.perf_counter()
    residuals = candidate_residuals(analysis_values, name)
    metrics = residual_metrics(residuals, time.perf_counter() - started)
    metrics["predictor"] = name
    rows.append(metrics)
predictor_baselines = display_metric_table(rows, sort_by="entropy_bits_per_symbol")

## 5. Evaluate predictors on identical partitions

The production residuals are split into train, validation, and test partitions only for diagnostics. The codec itself does not train on the original values: encoder and decoder share a same deterministic rule.

In [ ]:
# The production predictor is evaluated through the same deterministic scan used by the codec.
# q is the quantized residual symbol at Delta = 2 * bound.
bound = 0.05
production_codec = HoapsWvpaCodec(field.shape, bound, missing_value="nan", outer_compress=False)
step = 2 * bound
prior = np.full(field.shape, 32.0, dtype=np.float32)
scan_started = time.perf_counter()
symbols, reconstructed_rows, _ = production_codec._causal_scan_encode(field, mask, prior, step)
scan_seconds = time.perf_counter() - scan_started

production_metrics = residual_metrics(symbols, scan_seconds)
production_metrics["predictor"] = "HOAPS reconstructed-neighbor: left8/top2/diagonals1/time1"
production_metrics["symbols"] = symbols.size
production_metrics["scan_seconds"] = scan_seconds
production_row = pd.DataFrame([production_metrics])
display(pd.concat([predictor_baselines, production_row], ignore_index=True).sort_values("entropy_bits_per_symbol"))

# Identical train/validation/test partitions are used for residual diagnostics.
partition_count = len(symbols)
parts = np.array_split(symbols, [partition_count // 2, 3 * partition_count // 4])
partition_table = pd.DataFrame([
    {"partition": name, **residual_metrics(part)}
    for name, part in zip(["train", "validation", "test"], parts)
])
display(partition_table)

In [ ]:
# Residual distribution and predictor comparison.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(symbols, bins=100, log=True, color="steelblue")
axes[0].set_title("HOAPS quantized residual symbols")
axes[0].set_xlabel("symbol q")
axes[0].set_ylabel("count (log scale)")
plot_table = pd.concat([predictor_baselines, production_row], ignore_index=True)
axes[1].barh(plot_table["predictor"], plot_table["entropy_bits_per_symbol"], color="darkorange")
axes[1].set_title("Residual entropy")
axes[1].set_xlabel("bits per symbol")
plt.tight_layout()
plt.show()

## 6. Select the best predictor

Use a constrained rule rather than selecting on one statistic:

1. Reject predictors that violate the error tolerance or cannot be replayed deterministically.
2. Prefer the smallest measured encoded residual payload, not merely the smallest MAE.
3. Use residual entropy, RMSE, p99 error, runtime, and memory as diagnostics.
4. Validate the choice on held-out partitions and representative fields.

For the current reference field, the left-heavy HOAPS predictor is selected because it minimizes the measured entropy-coded residual size while preserving deterministic decoding.

## 7. Compare entropy coders

The project’s lossless entropy layer chooses per-block `RAW`, static byte-rANS (`RANGE`), or context-adaptive rANS (`CTX`). Huffman and arithmetic coding are useful external baselines, but they are not dependencies of this repository; the reproducible comparison below measures the implemented modes and reports the theoretical raw/varint baselines. Every selected representation must decode to the exact same symbols.

In [ ]:
def varint_size(values):
    values = np.asarray(values, dtype=np.uint64)
    sizes = np.ones(values.size, dtype=np.int64)
    remaining = values.copy()
    for _ in range(9):
        remaining >>= 7
        sizes += remaining != 0
    return int(sizes.sum())

def entropy_mode_counts(payload):
    if not payload:
        return {}
    n_blocks = int.from_bytes(payload[:4], "little")
    mode_bytes = payload[4:4 + n_blocks]
    names = {0: "RAW", 1: "RANGE", 2: "CTX"}
    return pd.Series([names.get(byte & 0x0F, "unknown") for byte in mode_bytes]).value_counts().to_dict()

zigzag_symbols = ((symbols << 1) ^ (symbols >> 63)).astype(np.uint64)
raw_i32_bytes = symbols.astype("<i4").nbytes
varint_bytes = varint_size(zigzag_symbols)
varint_payload = entropy.varint_encode(zigzag_symbols)
entropy_started = time.perf_counter()
entropy_payload = entropy.encode_symbols(symbols)
entropy_seconds = time.perf_counter() - entropy_started
decoded_symbols = entropy.decode_symbols(entropy_payload, len(symbols))
assert np.array_equal(symbols, decoded_symbols)

mode_counts = entropy_mode_counts(entropy_payload)
entropy_table = pd.DataFrame([
    {"coder_or_baseline": "RAW int32", "encoded_bytes": raw_i32_bytes, "overhead_note": "no model table"},
    {"coder_or_baseline": "Zigzag LEB128", "encoded_bytes": varint_bytes, "overhead_note": "no entropy model"},
    {"coder_or_baseline": "Project RAW/RANGE/CTX", "encoded_bytes": len(entropy_payload), "overhead_note": str(mode_counts)},
])
entropy_table["bits_per_symbol"] = 8 * entropy_table["encoded_bytes"] / len(symbols)
entropy_table["relative_to_raw_%"] = 100 * entropy_table["encoded_bytes"] / raw_i32_bytes
display(entropy_table)
print(f"entropy encode time: {entropy_seconds:.3f}s; exact decode: {np.array_equal(symbols, decoded_symbols)}")

z = zstd.ZstdCompressor(level=3)
entropy_table["zstd_bytes"] = [
    len(z.compress(symbols.astype("<i4").tobytes())),
    len(z.compress(varint_payload)),
    len(z.compress(entropy_payload)),
]
display(entropy_table[["coder_or_baseline", "encoded_bytes", "zstd_bytes", "bits_per_symbol"]])

## 8. Build the compression pipeline

The implemented `HWPC` container is versioned and stores:

- magic, container version, flags, and scan model version;
- JSON metadata: shape, dtype, missing-value policy, bound, quantization step, and repair metrics;
- a lossless mask payload;
- an entropy-coded residual payload;
- a CRC-32 integrity check.

Mask and residual payloads are compressed independently with Zstandard only when that reduces their size. This avoids expanding an already compact entropy stream.

## 9. Compress and decompress with the software

The public API is `HoapsWvpaCodec(shape, error_bound, missing_value, outer_compress)`. Encoding returns container bytes; decoding accepts those bytes and reconstructs a `float32` array. The next cell captures configuration, timing, payload sizes, and verification results.

In [ ]:
def benchmark_codec(field, bound, outer_compress=True):
    codec = HoapsWvpaCodec(
        shape=field.shape,
        error_bound=bound,
        missing_value="nan",
        outer_compress=outer_compress,
    )
    started = time.perf_counter()
    encoded = codec.encode(field)
    encode_seconds = time.perf_counter() - started
    started = time.perf_counter()
    decoded = codec.decode(encoded)
    decode_seconds = time.perf_counter() - started

    original_valid = np.isfinite(field)
    decoded_valid = np.isfinite(decoded)
    error = np.abs(decoded[original_valid].astype(np.float64) - field[original_valid].astype(np.float64))
    container = read_container(encoded)
    metrics = container.header_extra.get("metrics", {})
    return {
        "bound": bound,
        "outer_compress": outer_compress,
        "config": codec.get_config(),
        "original_bytes": field.nbytes,
        "compressed_bytes": len(encoded),
        "cr": field.nbytes / len(encoded),
        "reduction_pct": 100 * (1 - len(encoded) / field.nbytes),
        "encode_s": encode_seconds,
        "decode_s": decode_seconds,
        "encode_MB_s": field.nbytes / 1e6 / encode_seconds,
        "decode_MB_s": field.nbytes / 1e6 / decode_seconds,
        "max_abs_error": float(error.max()) if error.size else 0.0,
        "mae": float(error.mean()) if error.size else 0.0,
        "rmse": float(np.sqrt(np.mean(error ** 2))) if error.size else 0.0,
        "violations": int(np.sum(error > bound)),
        "mask_identical": bool(np.array_equal(original_valid, decoded_valid)),
        "payload_bytes": metrics.get("payload_size"),
        "mask_payload_bytes": len(container.mask_payload),
        "residual_payload_bytes": len(container.residual_payload),
        "model_version": container.model_version,
    }

results = []
for current_bound in (0.01, 0.05, 0.2):
    results.append(benchmark_codec(field, current_bound, outer_compress=True))
    results.append(benchmark_codec(field, current_bound, outer_compress=False))
results_table = pd.DataFrame(results)
display(results_table[["bound", "outer_compress", "original_bytes", "compressed_bytes", "cr", "encode_s", "decode_s", "max_abs_error", "violations", "mask_identical", "mask_payload_bytes", "residual_payload_bytes"]])

## 10. Validate reconstruction and report achieved CR

A valid run must preserve the missing mask exactly. For positive bounds, every valid-cell error must satisfy `abs(decoded - original) <= bound`. The achieved CR is computed from the complete container size, including metadata, payload lengths, and CRC.

In [ ]:
verified = results_table[results_table["outer_compress"]].copy()
assert (verified["violations"] == 0).all()
assert verified["mask_identical"].all()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for outer_value, group in results_table.groupby("outer_compress"):
    label = "Zstandard outer pass" if outer_value else "Outer pass disabled"
    axes[0].plot(group["bound"], group["cr"], marker="o", label=label)
axes[0].set_xscale("log")
axes[0].set_xlabel("error bound")
axes[0].set_ylabel("compression ratio")
axes[0].set_title("CR versus allowed error")
axes[0].legend()

axes[1].bar(verified["bound"].astype(str), verified["residual_payload_bytes"], label="residual")
axes[1].bar(verified["bound"].astype(str), verified["mask_payload_bytes"], bottom=verified["residual_payload_bytes"], label="mask")
axes[1].set_xlabel("error bound")
axes[1].set_ylabel("decoded payload bytes")
axes[1].set_title("Payload contribution")
axes[1].legend()
plt.tight_layout()
plt.show()

display(verified[["bound", "cr", "reduction_pct", "encode_s", "decode_s", "encode_MB_s", "decode_MB_s", "max_abs_error", "mae", "rmse"]])

## Conclusions

On this reference field, the largest contributors are:

- **Predictor:** the longitude-local reconstructed-neighbor stencil lowers residual entropy and residual payload size; predictor quality should be selected using measured entropy-coded bytes, not MAE alone.
- **Entropy coder:** the project’s per-block `RAW`/`RANGE`/`CTX` selection is lossless and adapts to local symbol distributions; exact symbol equality is verified after decoding.
- **Missing values:** the explicit mask preserves the grid and is encoded separately with RLE/bitpacking plus optional Zstandard. It is measurable but smaller than the residual contribution.
- **End-to-end result:** with the current implementation and reference dataset, the Zstandard-backed codec achieves approximately `12.75x`, `18.53x`, and `28.74x` CR at bounds `0.01`, `0.05`, and `0.2`, respectively. These values are benchmark results and should be regenerated on new fields and hardware.

For new datasets, rerun the predictor sweep, inspect validation/test residual entropy, compare complete encoded sizes, and reject any candidate that cannot be replayed deterministically, exact in its lossless stage, or within the configured error bound.